# Phase 5 — Econometrics & Robustness

**TL;DR.** Four pre-specified models absorb store×UPC and calendar-week fixed effects and use two-way entity/week clustered uncertainty. Documented-promotion transitions have large conditional associations with price-change probability and magnitude; decreases are larger and much more likely to reverse. All interpretations are associational. The corrected `PRICE_HEX`, bounded robustness, reconciliation, source-integrity, and deterministic-rerun checks pass. Final gate: **PASS WITH CONDITIONS**.

## Context & methods

This notebook is a lightweight, reproducible review companion. It reads generated Phase 5 outputs rather than re-estimating 4.5 million rows. Rebuild the analytical outputs from the repository root with `python src/econometrics.py --overwrite`; run twice after code or methodology changes to validate deterministic hashes.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
OUT = ROOT / 'data/processed/econometrics'
FIG = ROOT / 'report/figures'
pd.set_option('display.max_colwidth', 120)
manifest = json.loads((OUT / 'phase5_manifest.json').read_text())
sample = json.loads((OUT / 'econometric_sample_manifest.json').read_text())
assert manifest['phase5_closed'] and manifest['phase5_gate'] == 'PASS WITH CONDITIONS'
assert all(manifest['validations'].values())
manifest['phase5_gate'], manifest['pipeline_version'], len(manifest['validations'])

## Data and model contracts

Model 1 uses every approved consecutive comparison. Models 2–3 use changed prices. Model 4 additionally requires a valid following consecutive week. No model crosses a calendar gap or reintroduces an excluded non-food UPC.

In [ ]:
spec = pd.read_csv(OUT / 'model_specification_summary.csv')
display(spec[['model','dependent_variable','sample','main_regressors','reference_category','fixed_effects','clustering','N','entity_count','week_count']])
display(pd.Series(sample['phase4_reconciliation'], name='exact count').to_frame())

## Main results

Reported coefficients are percentage points for binary outcomes and log points for magnitude outcomes. Confidence intervals use primary two-way clustered standard errors.

In [ ]:
keep = ['term','coefficient_scaled','standard_error_two_way_scaled','ci_lower_two_way_scaled','ci_upper_two_way_scaled','scaled_unit','N','entity_count']
for filename in ['model1_price_change.csv','model2_change_magnitude.csv','model3_asymmetry_magnitude.csv','model4_reversal_probability.csv']:
    frame = pd.read_csv(OUT / filename)
    primary = frame.loc[frame['specification'].eq('primary'), keep].copy()
    print(filename)
    display(primary)

In [ ]:
for filename in ['15_promotion_association_coefficients.png','16_price_asymmetry_coefficients.png','17_reversal_probability_comparison.png','18_geks_jevons_difference.png']:
    display(Image(filename=str(FIG / filename), width=900))

## Robustness and validation

The robustness set was fixed before inspecting coefficients: corrected `PRICE_HEX`, top 0.1% magnitude exclusion, unknown-transition exclusion, a minimum-100 entity summary, and entity-only versus two-way clustered uncertainty. Primary samples remain unchanged.

In [ ]:
precision = sample['robustness_samples']['PRICE_HEX']
display(pd.Series(precision, name='corrected PRICE_HEX diagnostic').to_frame())
display(pd.Series(sample['synthetic_two_way_cluster_test'], name='two-way covariance validation').to_frame())
robustness = pd.read_csv(OUT / 'robustness_summary.csv')
display(robustness.groupby('robustness_check', as_index=False).agg(rows=('term','size'), all_directions_stable=('same_direction','all'), primary_N_min=('primary_N','min'), robustness_N_min=('robustness_N','min')))

## Hypothesis evidence and gate

H1 uses a dependence-aware interval rather than an iid test. H3 uses 10,000 seeded moving-block replications within each contiguous segment and never crosses a globally missing week. Labels emphasize economic magnitude and limitations rather than significance stars.

In [ ]:
h1 = sample['h1_two_way_cluster_interval']
h3 = sample['h3_block_bootstrap']
print(f"H1 frequency: {100*h1['mean']:.4f}% (95% two-way cluster interval {100*h1['ci_lower']:.4f}% to {100*h1['ci_upper']:.4f}%)")
print(f"H3 weekly GEKS−Jevons mean: {h3['observed_mean_weekly_difference_pp']:.4f} pp (block-bootstrap interval {h3['bootstrap_ci_lower_pp']:.4f} to {h3['bootstrap_ci_upper_pp']:.4f})")
display(pd.read_csv(OUT / 'hypothesis_summary.csv'))
print('Gate:', manifest['phase5_gate'])


## Takeaways

1. The fixed-effects estimates preserve the large promotion-transition associations observed descriptively.
2. Decreases are conditionally larger and more likely to reverse than increases.
3. Two-way clustered uncertainty is materially wider than entity-only uncertainty, so common weekly dependence cannot be ignored.
4. H3 is segment-dependent: cumulative GEKS–Jevons differences are material, while the mean weekly-difference interval crosses zero.
5. The econometric analysis is complete with the documented conditions; all interpretations remain associational rather than causal.